# Train EBM and GAMI-Net on 300k+VAE

## Install requirements

In [ ]:
%pip install -q --upgrade pip
%pip install -q "numpy<2" "pandas>=2.0,<2.3" "scikit-learn>=1.3,<1.6" matplotlib cloudpickle interpret
%pip install -q "tensorflow-cpu==2.15.*" "tensorflow-lattice<3" gaminet

## Imports

In [ ]:
import gc
import json
import pickle
import warnings
from pathlib import Path

import cloudpickle
import numpy as np
import pandas as pd
from IPython.display import display
from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

warnings.filterwarnings("ignore")


def resolve_existing(paths):
    for raw_path in paths:
        path = Path(raw_path)
        if path.exists():
            return path
    raise FileNotFoundError("None of these paths exist: " + ", ".join(str(p) for p in paths))


def batched_predict(model, X, batch_size=50_000):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch = X.iloc[start:stop] if hasattr(X, "iloc") else X[start:stop]
        preds.append(np.asarray(model.predict(batch)).astype(int))
    return np.concatenate(preds, axis=0)


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred).tolist(),
    }


def class_ratio(labels):
    labels = np.asarray(labels).astype(int)
    counts = np.bincount(labels, minlength=2)
    total = counts.sum()
    return {
        "class_0_count": int(counts[0]),
        "class_1_count": int(counts[1]),
        "class_0_ratio": float(counts[0] / total),
        "class_1_ratio": float(counts[1] / total),
    }


def build_error_subset_report(subset_name, reference_pred, peer_pred, y_true, mask):
    count = int(mask.sum())
    if count == 0:
        return {
            "subset_name": subset_name,
            "row_count": 0,
            "share_of_test_set": 0.0,
            "fidelity_to_reference": None,
            "peer_vs_true_label": None,
            "peer_matches_reference_count": 0,
            "peer_matches_reference_rate": None,
            "peer_matches_true_label_count": 0,
            "peer_matches_true_label_rate": None,
        }

    reference_subset = reference_pred[mask]
    peer_subset = peer_pred[mask]
    y_subset = y_true[mask]
    peer_matches_reference = peer_subset == reference_subset
    peer_matches_truth = peer_subset == y_subset

    return {
        "subset_name": subset_name,
        "row_count": count,
        "share_of_test_set": float(count / len(y_true)),
        "fidelity_to_reference": compute_metrics(reference_subset, peer_subset),
        "peer_vs_true_label": compute_metrics(y_subset, peer_subset),
        "peer_matches_reference_count": int(peer_matches_reference.sum()),
        "peer_matches_reference_rate": float(np.mean(peer_matches_reference)),
        "peer_matches_true_label_count": int(peer_matches_truth.sum()),
        "peer_matches_true_label_rate": float(np.mean(peer_matches_truth)),
    }


def build_teacher_error_report(teacher_pred, peer_pred, y_true):
    teacher_error_mask = teacher_pred != y_true
    teacher_fp_mask = (teacher_pred == 1) & (y_true == 0)
    teacher_fn_mask = (teacher_pred == 0) & (y_true == 1)
    return {
        "teacher_misclassified_count": int(teacher_error_mask.sum()),
        "teacher_misclassified_rate": float(np.mean(teacher_error_mask)),
        "misclassified_fidelity": build_error_subset_report("teacher_misclassified", teacher_pred, peer_pred, y_true, teacher_error_mask),
        "teacher_false_positive_subset": build_error_subset_report("teacher_false_positives", teacher_pred, peer_pred, y_true, teacher_fp_mask),
        "teacher_false_negative_subset": build_error_subset_report("teacher_false_negatives", teacher_pred, peer_pred, y_true, teacher_fn_mask),
    }


def build_ebm(feature_names, random_state):
    return ExplainableBoostingClassifier(
        feature_names=feature_names,
        interactions=25,
        validation_size=0.15,
        outer_bags=8,
        inner_bags=0,
        learning_rate=0.03,
        max_rounds=5000,
        early_stopping_rounds=100,
        max_bins=256,
        max_interaction_bins=64,
        n_jobs=-1,
        random_state=random_state,
    )


def ensure_gaminet_dependencies():
    import tensorflow as tf
    from gaminet import GAMINet
    return tf, GAMINet


def build_meta_info_and_scale(X_train, X_other_list):
    feature_names = X_train.columns.tolist()
    train_np = X_train.to_numpy(dtype=np.float32, copy=True)
    other_np_list = [frame.to_numpy(dtype=np.float32, copy=True) for frame in X_other_list]

    train_scaled = np.zeros_like(train_np, dtype=np.float32)
    other_scaled_list = [np.zeros_like(arr, dtype=np.float32) for arr in other_np_list]
    meta_info = {}

    for idx, feature_name in enumerate(feature_names):
        scaler = MinMaxScaler(feature_range=(0.0, 1.0))
        train_col = train_np[:, [idx]]
        scaler.fit(train_col)
        train_scaled[:, [idx]] = scaler.transform(train_col).astype(np.float32)
        for arr_idx, arr in enumerate(other_np_list):
            other_scaled_list[arr_idx][:, [idx]] = scaler.transform(arr[:, [idx]]).astype(np.float32)
        meta_info[feature_name] = {"type": "continuous", "scaler": scaler}

    meta_info["label"] = {"type": "target"}
    return train_scaled, other_scaled_list, meta_info


def build_gaminet(meta_info, random_state, interact_num=5, batch_size=512, main_effect_epochs=50, interaction_epochs=50, tuning_epochs=20):
    tf, GAMINet = ensure_gaminet_dependencies()
    tf.random.set_seed(random_state)
    np.random.seed(random_state)
    return GAMINet(
        meta_info=meta_info,
        interact_num=interact_num,
        interact_arch=[40] * 5,
        subnet_arch=[40] * 5,
        batch_size=batch_size,
        task_type="Classification",
        activation_func=tf.nn.relu,
        main_effect_epochs=main_effect_epochs,
        interaction_epochs=interaction_epochs,
        tuning_epochs=tuning_epochs,
        lr_bp=[0.0001, 0.0001, 0.0001],
        early_stop_thres=[30, 30, 20],
        heredity=True,
        loss_threshold=0.0,
        reg_clarity=0.1,
        mono_increasing_list=[],
        mono_decreasing_list=[],
        verbose=True,
        val_ratio=0.15,
        random_state=random_state,
    )


def gaminet_predict_labels(model, X, batch_size=8192):
    preds = []
    for start in range(0, len(X), batch_size):
        stop = min(start + batch_size, len(X))
        batch_pred = np.asarray(model.predict(X[start:stop]), dtype=np.float32).reshape(-1)
        if np.nanmin(batch_pred) < 0.0 or np.nanmax(batch_pred) > 1.0:
            batch_pred = 1.0 / (1.0 + np.exp(-batch_pred))
        preds.append((batch_pred >= 0.5).astype(np.int32))
    return np.concatenate(preds, axis=0)


def save_json(payload, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def cleanup_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()

## Paths and config

In [ ]:
SEED = 1237
VAE_SAMPLE_SIZE = 100_000
LOCAL_SAMPLE_SIZE = 300_000
OUTPUT_ROOT = Path("codespace_300k_vae_run")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
TEACHER_DATASET_DIR = OUTPUT_ROOT / "teacher_datasets"
TEACHER_DATASET_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = OUTPUT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

processed_path = resolve_existing([
    "csv/processed/processed_dataset_with_split.csv",
    "processed_csv/processed_dataset_with_split.csv",
    "artifacts/processed_dataset_with_split.csv",
])

local_path = resolve_existing([
    "csv/synthetic/synthetic_local_permutation_300k.csv",
    "artifacts/synthetic_local_permutation_300k.csv",
])

vae_path = resolve_existing([
    "csv/synthetic/synthetic_vae_ld20_warm10_temp0p85_filtered_300k.csv",
    "artifacts/synthetic_vae_ld20_warm10_temp0p85_filtered_300k.csv",
])

teacher_model_paths = {
    "random_forest": [
        "artifacts/random_forest.pkl",
        "random_forest.pkl",
    ],
    "deep_neural_net": [
        "artifacts/deep_neural_net.pkl",
        "deep_neural_net.pkl",
    ],
}

teacher_dataset_candidates = {
    "random_forest": [
        "artifact_2/surrogates/surrogate_training_random_forest.csv",
        "artifact_4/surrogates/surrogate_training_real_train_only_92602_random_forest.csv",
    ],
    "deep_neural_net": [
        "artifact_2/surrogates/surrogate_training_deep_neural_net.csv",
        "artifact_4/surrogates/surrogate_training_real_train_only_92602_deep_neural_net.csv",
    ],
}

teacher_test_cache_candidates = {
    "random_forest": [
        "artifact_4/surrogates/teacher_real_test_predictions_random_forest.npy",
        "artifact_3/surrogates/teacher_real_test_predictions_random_forest.npy",
        "artifact_2/surrogates/teacher_real_test_predictions_random_forest.npy",
        "artifacts/surrogates/teacher_real_test_predictions_random_forest.npy",
        "model/teacher_real_test_predictions_random_forest.npy",
    ],
    "deep_neural_net": [
        "artifact_4/surrogates/teacher_real_test_predictions_deep_neural_net.npy",
        "artifact_3/surrogates/teacher_real_test_predictions_deep_neural_net.npy",
        "artifact_2/surrogates/teacher_real_test_predictions_deep_neural_net.npy",
        "artifacts/surrogates/teacher_real_test_predictions_deep_neural_net.npy",
        "model/teacher_real_test_predictions_deep_neural_net.npy",
    ],
}

print(f"Processed dataset: {processed_path}")
print(f"Local permutation CSV: {local_path}")
print(f"VAE CSV: {vae_path}")
print(f"Notebook seed: {SEED}")

## Load processed test split

In [ ]:
processed = pd.read_csv(processed_path)
feature_cols = [col for col in processed.columns if col not in {"label", "split"}]
test_df = processed[processed["split"] == "test"].copy()
X_test = test_df[feature_cols].astype(np.float32)
y_test = test_df["label"].to_numpy(dtype=np.int32, copy=True)
print({"test_rows": len(X_test), "feature_count": len(feature_cols)})

## Load or build teacher-labeled 300k+VAE datasets

In [ ]:
teacher_models = {}
teacher_test_predictions = {}
teacher_training_datasets = {}

for teacher_name, cache_candidates in teacher_test_cache_candidates.items():
    existing_cache = next((Path(p) for p in cache_candidates if Path(p).exists()), None)
    if existing_cache is not None:
        teacher_test_predictions[teacher_name] = np.load(existing_cache).astype(np.int32)
        print(f"Loaded cached test predictions for {teacher_name}: {existing_cache}")

for teacher_name, candidates in teacher_dataset_candidates.items():
    existing_dataset = next((Path(p) for p in candidates if Path(p).exists()), None)
    if existing_dataset is not None:
        df = pd.read_csv(existing_dataset).astype(np.float32)
        if list(df.columns) != feature_cols + ["label"]:
            raise ValueError(f"Unexpected schema for {existing_dataset}")
        teacher_training_datasets[teacher_name] = df
        print(f"Loaded teacher-labeled dataset for {teacher_name}: {existing_dataset}")

need_generated_dataset = any(name not in teacher_training_datasets for name in ["random_forest", "deep_neural_net"])
need_generated_test_preds = any(name not in teacher_test_predictions for name in ["random_forest", "deep_neural_net"])

if need_generated_dataset or need_generated_test_preds:
    for teacher_name, candidates in teacher_model_paths.items():
        teacher_model_path = resolve_existing(candidates)
        with teacher_model_path.open("rb") as handle:
            teacher_models[teacher_name] = pickle.load(handle)
        print(f"Loaded teacher model for {teacher_name}: {teacher_model_path}")

if need_generated_dataset:
    local_df = pd.read_csv(local_path).astype(np.float32)
    vae_df = pd.read_csv(vae_path).astype(np.float32)
    if list(local_df.columns) != feature_cols:
        raise ValueError("Local permutation schema does not match processed feature schema")
    if list(vae_df.columns) != feature_cols:
        raise ValueError("VAE schema does not match processed feature schema")
    local_part = local_df.iloc[:LOCAL_SAMPLE_SIZE].copy()
    vae_part = vae_df.sample(n=VAE_SAMPLE_SIZE, random_state=SEED).reset_index(drop=True)
    combined_df = pd.concat([local_part, vae_part], axis=0, ignore_index=True)
    print({"combined_rows": len(combined_df), "local_rows": len(local_part), "vae_rows": len(vae_part)})
    for teacher_name in ["random_forest", "deep_neural_net"]:
        if teacher_name in teacher_training_datasets:
            continue
        labels = batched_predict(teacher_models[teacher_name], combined_df)
        teacher_df = combined_df.copy()
        teacher_df["label"] = labels
        out_path = TEACHER_DATASET_DIR / f"surrogate_training_300k_plus_vae_{teacher_name}.csv"
        teacher_df.to_csv(out_path, index=False, float_format="%.6g")
        teacher_training_datasets[teacher_name] = teacher_df
        print(f"Saved teacher-labeled dataset for {teacher_name}: {out_path}")
        print(class_ratio(labels))

if need_generated_test_preds:
    for teacher_name in ["random_forest", "deep_neural_net"]:
        if teacher_name in teacher_test_predictions:
            continue
        teacher_test_predictions[teacher_name] = batched_predict(teacher_models[teacher_name], X_test)
        cache_path = OUTPUT_ROOT / f"teacher_real_test_predictions_{teacher_name}.npy"
        np.save(cache_path, teacher_test_predictions[teacher_name])
        print(f"Saved test predictions for {teacher_name}: {cache_path}")

for teacher_name, df in teacher_training_datasets.items():
    print(teacher_name, df.shape)

## Train EBM surrogates

In [ ]:
ebm_results = {}

for teacher_name, teacher_df in teacher_training_datasets.items():
    y_teacher = teacher_df["label"].to_numpy(dtype=np.int32, copy=True)
    X_teacher = teacher_df[feature_cols].astype(np.float32)
    X_train, X_holdout, y_train, y_holdout = train_test_split(
        X_teacher,
        y_teacher,
        test_size=0.30,
        random_state=SEED,
        stratify=y_teacher,
    )
    model = build_ebm(feature_cols, random_state=SEED)
    model.fit(X_train, y_train)
    holdout_pred = batched_predict(model, X_holdout)
    test_pred = batched_predict(model, X_test)
    model_path = MODEL_DIR / f"ebm_300k_plus_vae_{teacher_name}.pkl"
    with model_path.open("wb") as handle:
        cloudpickle.dump(model, handle)
    error_report = build_teacher_error_report(teacher_test_predictions[teacher_name], test_pred, y_test)
    ebm_results[teacher_name] = {
        "teacher": teacher_name,
        "surrogate": "ebm",
        "rows": {
            "teacher_dataset_total": int(len(teacher_df)),
            "holdout": int(len(X_holdout)),
            "real_test": int(len(X_test)),
        },
        "class_ratio": class_ratio(y_teacher),
        "holdout_fidelity": compute_metrics(y_holdout, holdout_pred),
        "real_test_fidelity_to_teacher": compute_metrics(teacher_test_predictions[teacher_name], test_pred),
        "real_test_accuracy_to_true_label": compute_metrics(y_test, test_pred),
        "error_fidelity": error_report,
        "model_path": str(model_path),
    }
    print(teacher_name, ebm_results[teacher_name]["real_test_fidelity_to_teacher"]["accuracy"], ebm_results[teacher_name]["error_fidelity"]["misclassified_fidelity"]["fidelity_to_reference"]["accuracy"])
    cleanup_memory(X_train, X_holdout, y_train, y_holdout, holdout_pred, test_pred, model)

save_json(ebm_results, OUTPUT_ROOT / "ebm_300k_plus_vae_summary.json")

## Train GAMI-Net surrogates

In [ ]:
tf, _ = ensure_gaminet_dependencies()
tf.get_logger().setLevel("ERROR")
gaminet_results = {}

for teacher_name, teacher_df in teacher_training_datasets.items():
    y_teacher = teacher_df["label"].to_numpy(dtype=np.int32, copy=True)
    X_teacher = teacher_df[feature_cols].astype(np.float32)
    X_train_df, X_holdout_df, y_train, y_holdout = train_test_split(
        X_teacher,
        y_teacher,
        test_size=0.30,
        random_state=SEED,
        stratify=y_teacher,
    )
    X_train_scaled, other_scaled, meta_info = build_meta_info_and_scale(X_train_df, [X_holdout_df, X_test])
    X_holdout_scaled, X_test_scaled = other_scaled
    y_train_gami = y_train.astype(np.float32).reshape(-1, 1)
    model = build_gaminet(meta_info, random_state=SEED)
    model.fit(X_train_scaled, y_train_gami)
    holdout_pred = gaminet_predict_labels(model, X_holdout_scaled)
    test_pred = gaminet_predict_labels(model, X_test_scaled)
    teacher_output_dir = MODEL_DIR / f"gaminet_300k_plus_vae_{teacher_name}"
    teacher_output_dir.mkdir(parents=True, exist_ok=True)
    model_name = f"gaminet_300k_plus_vae_{teacher_name}"
    model.save(folder=str(teacher_output_dir) + "/", name=model_name)
    model_pickle_path = teacher_output_dir / f"{model_name}.pickle"
    error_report = build_teacher_error_report(teacher_test_predictions[teacher_name], test_pred, y_test)
    gaminet_results[teacher_name] = {
        "teacher": teacher_name,
        "surrogate": "gaminet",
        "rows": {
            "teacher_dataset_total": int(len(teacher_df)),
            "holdout": int(len(X_holdout_df)),
            "real_test": int(len(X_test)),
        },
        "class_ratio": class_ratio(y_teacher),
        "holdout_fidelity": compute_metrics(y_holdout, holdout_pred),
        "real_test_fidelity_to_teacher": compute_metrics(teacher_test_predictions[teacher_name], test_pred),
        "real_test_accuracy_to_true_label": compute_metrics(y_test, test_pred),
        "error_fidelity": error_report,
        "model_path": str(model_pickle_path),
    }
    print(teacher_name, gaminet_results[teacher_name]["real_test_fidelity_to_teacher"]["accuracy"], gaminet_results[teacher_name]["error_fidelity"]["misclassified_fidelity"]["fidelity_to_reference"]["accuracy"])
    cleanup_memory(
        X_train_df, X_holdout_df, y_train, y_holdout, y_train_gami,
        X_train_scaled, X_holdout_scaled, X_test_scaled,
        holdout_pred, test_pred, model,
    )

save_json(gaminet_results, OUTPUT_ROOT / "gaminet_300k_plus_vae_summary.json")

## Save summary table

In [ ]:
rows = []
for surrogate_name, result_dict in [("ebm", ebm_results), ("gaminet", gaminet_results)]:
    for teacher_name, payload in result_dict.items():
        mis = payload["error_fidelity"]["misclassified_fidelity"]
        fp = payload["error_fidelity"]["teacher_false_positive_subset"]
        fn = payload["error_fidelity"]["teacher_false_negative_subset"]
        rows.append({
            "surrogate": surrogate_name,
            "teacher": teacher_name,
            "teacher_dataset_rows": payload["rows"]["teacher_dataset_total"],
            "holdout_fidelity_accuracy": payload["holdout_fidelity"]["accuracy"],
            "holdout_fidelity_f1": payload["holdout_fidelity"]["f1"],
            "overall_fidelity_accuracy": payload["real_test_fidelity_to_teacher"]["accuracy"],
            "overall_fidelity_f1": payload["real_test_fidelity_to_teacher"]["f1"],
            "surrogate_test_accuracy": payload["real_test_accuracy_to_true_label"]["accuracy"],
            "surrogate_test_f1": payload["real_test_accuracy_to_true_label"]["f1"],
            "teacher_error_count": payload["error_fidelity"]["teacher_misclassified_count"],
            "teacher_error_rate": payload["error_fidelity"]["teacher_misclassified_rate"],
            "error_fidelity_accuracy": mis["fidelity_to_reference"]["accuracy"],
            "error_fidelity_f1": mis["fidelity_to_reference"]["f1"],
            "error_match_truth_rate": mis["peer_matches_true_label_rate"],
            "teacher_fp_count": fp["row_count"],
            "fp_error_fidelity": fp["peer_matches_reference_rate"],
            "fp_match_truth_rate": fp["peer_matches_true_label_rate"],
            "teacher_fn_count": fn["row_count"],
            "fn_error_fidelity": fn["peer_matches_reference_rate"],
            "fn_match_truth_rate": fn["peer_matches_true_label_rate"],
        })

summary_df = pd.DataFrame(rows).sort_values(["surrogate", "teacher"]).reset_index(drop=True)
summary_path = OUTPUT_ROOT / "surrogate_300k_plus_vae_summary.csv"
summary_df.to_csv(summary_path, index=False)
display(summary_df)
print(f"Saved summary CSV: {summary_path}")